<a href="https://colab.research.google.com/github/Karsuman4298/AI_project/blob/main/EMCAD_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Create data folder
!mkdir -p data/polyp

# Best way: Mount Google Drive and copy, or download directly if they provide links
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Global fix - replace in training scripts if needed
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

Using device: cuda


### MSDC: Multi-scale Depth-wise Convolution (Core Innovation)

**Purpose**:  
MSDC is the foundational building block of EMCAD. It captures **multi-scale contextual information** very efficiently.

**How it Works**:
- Runs **three parallel branches** with different kernel sizes: **1×1, 3×3, and 5×5**
- Each branch is a **depth-wise convolution** (applies filter independently per channel)
- Outputs of all branches are **added** together
- Followed by BatchNorm + ReLU6

**Why it is powerful**:
- 1×1 → Point-wise / channel information
- 3×3 → Local textures and edges
- 5×5 → Larger contextual structures (organs, lesions)
- Depth-wise design keeps parameters and FLOPs extremely low



In [ ]:
class MSDC(nn.Module):
    """Multi-Scale Depth-wise Convolution"""
    def __init__(self, channels):
        super().__init__()
        self.dw1 = nn.Conv2d(channels, channels, 1, padding=0, groups=channels)
        self.dw3 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels)
        self.dw5 = nn.Conv2d(channels, channels, 5, padding=2, groups=channels)

        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU6(inplace=True)

    def forward(self, x):
        x1 = self.relu(self.bn(self.dw1(x)))
        x3 = self.relu(self.bn(self.dw3(x)))
        x5 = self.relu(self.bn(self.dw5(x)))
        return x1 + x3 + x5

### EMCAD: Efficient Multi-scale Convolutional Attention Decoding

**EMCAD** is a lightweight decoder designed for medical image segmentation. It can be plugged into any hierarchical encoder (such as PVT, ResNet, Swin Transformer, etc.).

While the **encoder** extracts multi-scale features from the input image, the **EMCAD decoder** is responsible for progressively refining these features and producing the final segmentation mask. The decoder follows a U-Net-style structure but is heavily optimized for **high accuracy at very low computational cost**.

---

### Core Innovations in EMCAD Decoder

- **MSDC** (Multi-scale Depth-wise Convolutions)
- **MSCAM** (Multi-scale Convolutional Attention Module)
- **LGAG** (Large-kernel Grouped Attention Gate)
- **EUCB** (Efficient Up-convolution Block)

---

### Detailed Explanation of EUCB (Efficient Up-convolution Block)

**EUCB** is the **upsampling module** used in the EMCAD decoder. Its main purpose is to **progressively increase the spatial resolution** of feature maps while moving from deep (low-resolution, high-semantic) layers toward shallow (high-resolution) layers.

#### Operations Inside EUCB (Step-by-Step):

1. **Bilinear Upsampling**
   - Increases the height and width of the feature map by a factor of 2 (e.g., 32×32 → 64×64).
   - Uses **bilinear interpolation**: calculates new pixel values as a weighted average of the 4 nearest neighboring pixels.
   - **Advantages**: Very fast, smooth results, zero parameters.
   - **Limitation**: Purely mathematical (no learning), can introduce slight blurring.

2. **Depth-wise Convolution (3×3)**
   - Applies a separate 3×3 filter to **each input channel independently**.
   - Implemented in PyTorch using `groups = in_channels`.
   - **Advantages**: Extremely parameter-efficient (parameters scale linearly with channels instead of quadratically). Captures spatial patterns (edges, textures) per channel at very low cost.

3. **Batch Normalization + ReLU6**
   - Normalizes the features and introduces non-linearity for stable training.

4. **Point-wise Convolution (1×1)**
   - Uses 1×1 kernels to mix information **across different channels** and adjust the number of output channels.
   - **Advantages**: Very cheap way to combine channel-wise information and match dimensions for skip connection fusion.

---

### Complete EUCB Implementation


In [ ]:
from torch import nn
class EUCB(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dw = nn.Conv2d(in_channels, in_channels, 3, padding=1, groups=in_channels)
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.pw = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x):
        x = self.up(x)
        x = self.relu(self.bn(self.dw(x)))
        x = self.pw(x)
        return x

In [17]:
print(test)

EUCB(
  (up): Upsample(scale_factor=2.0, mode='bilinear')
  (dw): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3)
  (bn): BatchNorm2d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (pw): Conv2d(3, 1, kernel_size=(1, 1), stride=(1, 1))
)


### MSCAM: Multi-scale Convolutional Attention Module

**Purpose**:  
MSCAM is the **main refinement engine** of EMCAD. It suppresses irrelevant regions and highlights salient features using channel attention, spatial attention, and multi-scale context.

**Structure**:
1. **CAB** (Channel Attention Block) → Decides *which channels* are important
2. **SAB** (Spatial Attention Block) → Decides *where* to focus in the image
3. **MSCB** (Multi-scale Convolution Block) → Extracts rich multi-scale features

**Why it works**:
- Combines **"What"** (channel) and **"Where"** (spatial) attention
- Integrates multi-scale context efficiently
- Depth-wise operations keep it lightweight

In [ ]:
class MSCAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Channel Attention
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels//4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels//4, channels, 1),
            nn.Sigmoid()
        )
        # Spatial Attention
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )
        self.msdc = MSDC(channels)
        self.conv1x1 = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        # Channel Attention
        ca_weight = self.ca(x)
        x = x * ca_weight

        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        sa_weight = self.sa(torch.cat([avg_out, max_out], dim=1))
        x = x * sa_weight

        # Multi-scale Convolution
        x = self.msdc(x)
        x = self.conv1x1(x)
        return x

### LGAG: Large-kernel Grouped Attention Gate

**Purpose**:  
LGAG intelligently fuses features from the encoder skip connection with upsampled decoder features. It replaces the classic Attention Gate with a more effective and efficient version.

**How it Works**:  
- Takes two inputs: upsampled feature from the decoder (`x`) and skip connection from the encoder (`skip`).  
- Applies 3x3 grouped convolutions separately on both inputs.  
- Adds the two processed features together.  
- Applies Batch Normalization and ReLU activation.  
- Generates a soft attention gate using a 1x1 convolution followed by sigmoid.  
- Multiplies this gate with the decoder feature to produce the final refined output.

**Key Advantages over Standard Attention Gate**:  
- Larger receptive field (3x3 instead of 1x1) for better local context.  
- Uses grouped convolution, which keeps computational cost and parameters low.  
- Achieves superior fusion between high-level semantic information (from decoder) and fine-grained details (from encoder skip connections).

This module is very important in the cascaded decoder as it enables effective multi-scale feature fusion at every stage while maintaining overall efficiency.

In [31]:
class LGAG(nn.Module):
    """Large-kernel Grouped Attention Gate"""
    def __init__(self, channels):
        super().__init__()
        # 3x3 Grouped Convolutions
        self.conv_skip = nn.Conv2d(channels, channels, kernel_size=3,
                                   padding=1, groups=channels//4)
        self.conv_x = nn.Conv2d(channels, channels, kernel_size=3,
                                padding=1, groups=channels//4)

        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv_gate = nn.Conv2d(channels, 1, kernel_size=1)

    def forward(self, x, skip):
        """
        x: upsampled feature from decoder
        skip: skip connection from encoder
        """
        g = self.conv_skip(skip)
        x_feat = self.conv_x(x)
        fused = g + x_feat
        fused = self.relu(self.bn(fused))
        gate = torch.sigmoid(self.conv_gate(fused))
        return x_feat * gate